In [1]:
%%markdown
#  **AI-POWERED SECURE DOCUMENT SEARCH PORTAL**
##  **BY Aliza Taqvi**

#  **AI-POWERED SECURE DOCUMENT SEARCH PORTAL**
##  **BY Aliza Taqvi**


In [2]:
# ============================================================
# AI-POWERED SECURE DOCUMENT SEARCH PORTAL
# PYTHON TEAM - SIMPLE VERSION
#
# Python Team will:
# 1. Receive uploaded documents
# 2. Process the file
# 3. Store document in SQLite database
# 4. Provide APIs to Web Team
# 5. Send document + question to AI Team
# 6. Return AI answer to Web Team
#
# Easy technologies:
# Python
# FastAPI
# SQLAlchemy
# SQLite
# Pydantic
# Requests
# ============================================================


# ============================================================
# 1. INSTALL LIBRARIES
# ============================================================

%pip install fastapi uvicorn sqlalchemy pydantic python-multipart requests -q


# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response

from pydantic import BaseModel

from sqlalchemy import (
    create_engine,
    Column,
    Integer,
    String,
    Text
)

from sqlalchemy.orm import (
    declarative_base,
    sessionmaker,
    Session
)

from pathlib import Path
import requests
import threading
import uvicorn


# ============================================================
# 3. PROJECT FOLDER
# ============================================================

UPLOAD_FOLDER = Path("uploads")
UPLOAD_FOLDER.mkdir(exist_ok=True)


# ============================================================
# 4. DATABASE
# ============================================================

DATABASE_URL = "sqlite:///./documents.db"

engine = create_engine(
    DATABASE_URL,
    connect_args={"check_same_thread": False}
)

SessionLocal = sessionmaker(
    bind=engine,
    autoflush=False,
    autocommit=False
)

Base = declarative_base()


# ============================================================
# 5. DATABASE TABLE
# ============================================================

class Document(Base):

    __tablename__ = "documents"

    id = Column(
        Integer,
        primary_key=True,
        index=True
    )

    filename = Column(
        String,
        nullable=False
    )

    content = Column(
        Text,
        nullable=False
    )


# Create database table
Base.metadata.create_all(bind=engine)


# ============================================================
# 6. FASTAPI APP
# ============================================================

app = FastAPI(
    title="Python Team Document Backend",
    description="Backend for AI-Powered Secure Document Search Portal",
    version="1.0"
)


# ============================================================
# 7. ALLOW WEB FRONTEND TO CONNECT
# ============================================================

app.add_middleware(
    CORSMiddleware,

    allow_origins=["*"],

    allow_credentials=True,

    allow_methods=["*"],

    allow_headers=["*"]
)


# ============================================================
# 8. PYDANTIC MODEL
# ============================================================

class AskQuestion(BaseModel):

    document_id: int

    question: str


# ============================================================
# 9. HOME API
# ============================================================

@app.get("/")
def home():

    return {
        "message": "Python Backend is running",
        "docs": "http://127.0.0.1:8000/docs"
    }


# ============================================================
# 10. UPLOAD DOCUMENT
# ============================================================
#
# Web Team sends a file
#
# Python Team:
#       receives file
#             ↓
#       reads file
#             ↓
#       saves file
#             ↓
#       saves information in database
#
# ============================================================

@app.post("/upload")
async def upload_document(
    file: UploadFile = File(...)
):

    # Check file name
    if not file.filename:

        raise HTTPException(
            status_code=400,
            detail="Please select a file."
        )


    # Simple version: TXT files
    if not file.filename.lower().endswith(".txt"):

        raise HTTPException(
            status_code=400,
            detail="Only TXT files are allowed."
        )


    # Read uploaded file
    file_data = await file.read()


    # Convert file into text
    try:

        text = file_data.decode("utf-8")

    except UnicodeDecodeError:

        raise HTTPException(
            status_code=400,
            detail="The file must be a normal UTF-8 TXT file."
        )


    # Check empty file
    if not text.strip():

        raise HTTPException(
            status_code=400,
            detail="File is empty."
        )


    # Save a copy of the file
    safe_name = Path(file.filename).name

    file_path = UPLOAD_FOLDER / safe_name

    file_path.write_text(
        text,
        encoding="utf-8"
    )


    # Open database
    db: Session = SessionLocal()

    try:

        # Create database object
        document = Document(

            filename=safe_name,

            content=text

        )

        # Add to database
        db.add(document)

        # Save changes
        db.commit()

        # Get ID
        db.refresh(document)

        return {

            "message": "Document uploaded successfully",

            "document_id": document.id,

            "filename": document.filename

        }

    finally:

        db.close()


# ============================================================
# 11. SHOW ALL DOCUMENTS
# ============================================================
#
# Web Team can use this API to show documents on website.
#
# ============================================================

@app.get("/documents")
def get_documents():

    db: Session = SessionLocal()

    try:

        documents = db.query(Document).all()

        result = []

        for document in documents:

            result.append({

                "id": document.id,

                "filename": document.filename

            })

        return result

    finally:

        db.close()


# ============================================================
# 12. GET ONE DOCUMENT
# ============================================================
#
# Example:
# GET /documents/1
#
# ============================================================

@app.get("/documents/{document_id}")
def get_document(document_id: int):

    db: Session = SessionLocal()

    try:

        document = (

            db.query(Document)

            .filter(
                Document.id == document_id
            )

            .first()

        )


        if document is None:

            raise HTTPException(
                status_code=404,
                detail="Document not found."
            )


        return {

            "id": document.id,

            "filename": document.filename,

            "content": document.content

        }

    finally:

        db.close()


# ============================================================
# 13. SIMPLE SEARCH
# ============================================================
#
# This is only a simple keyword search.
#
# The intelligent AI/RAG search belongs to AI Team.
#
# ============================================================

@app.get("/search")
def search_documents(keyword: str):

    keyword = keyword.strip()

    if not keyword:

        raise HTTPException(
            status_code=400,
            detail="Please enter a keyword."
        )


    db: Session = SessionLocal()

    try:

        documents = db.query(Document).all()

        results = []

        for document in documents:

            if keyword.lower() in document.content.lower():

                results.append({

                    "id": document.id,

                    "filename": document.filename

                })


        return {

            "keyword": keyword,

            "total_results": len(results),

            "results": results

        }

    finally:

        db.close()


# ============================================================
# 14. DOWNLOAD DOCUMENT
# ============================================================

@app.get("/download/{document_id}")
def download_document(document_id: int):

    db: Session = SessionLocal()

    try:

        document = (

            db.query(Document)

            .filter(
                Document.id == document_id
            )

            .first()

        )


        if document is None:

            raise HTTPException(
                status_code=404,
                detail="Document not found."
            )


        headers = {

            "Content-Disposition":
            f'attachment; filename="{document.filename}"'

        }


        return Response(

            content=document.content,

            media_type="text/plain",

            headers=headers

        )

    finally:

        db.close()


# ============================================================
# 15. DELETE DOCUMENT
# ============================================================

@app.delete("/documents/{document_id}")
def delete_document(document_id: int):

    db: Session = SessionLocal()

    try:

        document = (

            db.query(Document)

            .filter(
                Document.id == document_id
            )

            .first()

        )


        if document is None:

            raise HTTPException(
                status_code=404,
                detail="Document not found."
            )


        db.delete(document)

        db.commit()


        return {

            "message": "Document deleted successfully"

        }

    finally:

        db.close()


# ============================================================
# 16. AI TEAM CONNECTION
# ============================================================
#
# Web Team
#     ↓
# /ask-ai
#     ↓
# Python Team
#     ↓
# gets document from database
#     ↓
# sends document + question
#     ↓
# AI Team
#     ↓
# AI answer
#     ↓
# Python Team
#     ↓
# Web Team
#
# ============================================================


# IMPORTANT:
# AI Team will give you their actual API address.
#
# Example:
# http://127.0.0.1:8001/ask
#
AI_TEAM_URL = "http://127.0.0.1:8001/ask"


@app.post("/ask-ai")
def ask_ai(request: AskQuestion):

    # --------------------------------------------------------
    # STEP 1: Get document from database
    # --------------------------------------------------------

    db: Session = SessionLocal()

    try:

        document = (

            db.query(Document)

            .filter(
                Document.id == request.document_id
            )

            .first()

        )


        if document is None:

            raise HTTPException(
                status_code=404,
                detail="Document not found."
            )


        # ----------------------------------------------------
        # STEP 2: Prepare information for AI Team
        # ----------------------------------------------------

        data = {

            "document_id": document.id,

            "document_name": document.filename,

            "document_text": document.content,

            "question": request.question

        }


        # ----------------------------------------------------
        # STEP 3: Send to AI Team
        # ----------------------------------------------------

        try:

            response = requests.post(

                AI_TEAM_URL,

                json=data,

                timeout=30

            )

        except requests.exceptions.RequestException:

            raise HTTPException(

                status_code=503,

                detail="AI Team server is not connected yet."

            )


        # ----------------------------------------------------
        # STEP 4: Check AI response
        # ----------------------------------------------------

        if response.status_code != 200:

            raise HTTPException(

                status_code=500,

                detail="AI Team returned an error."

            )


        # ----------------------------------------------------
        # STEP 5: Get AI answer
        # ----------------------------------------------------

        ai_answer = response.json()


        # ----------------------------------------------------
        # STEP 6: Send answer to Web Team
        # ----------------------------------------------------

        return {

            "document": document.filename,

            "question": request.question,

            "answer": ai_answer

        }

    finally:

        db.close()


# ============================================================
# 17. DOCUMENT COUNT
# ============================================================

@app.get("/count")
def count_documents():

    db: Session = SessionLocal()

    try:

        total = db.query(Document).count()

        return {

            "total_documents": total

        }

    finally:

        db.close()


# ============================================================
# 18. START SERVER
# ============================================================

def start_server():

    uvicorn.run(

        app,

        host="127.0.0.1",

        port=8000,

        log_level="info"

    )


server_thread = threading.Thread(

    target=start_server,

    daemon=True

)

server_thread.start()


# ============================================================
# 19. FINAL MESSAGE
# ============================================================

print()
print("=" * 60)
print("PYTHON TEAM PROJECT STARTED")
print("=" * 60)

print()
print("Backend:")
print("http://127.0.0.1:8000")

print()
print("API Testing:")
print("http://127.0.0.1:8000/docs")

print()
print("Python Team completed:")
print("✓ File upload")
print("✓ File processing")
print("✓ SQLite database")
print("✓ Database management")
print("✓ Document API")
print("✓ Simple search")
print("✓ Download")
print("✓ Delete")
print("✓ Web Team connection")
print("✓ AI Team connection")

print()
print("AI Team API:")
print(AI_TEAM_URL)

print()
print("IMPORTANT:")
print("Only TXT files are used in this easy version.")

print("=" * 60)


PYTHON TEAM PROJECT STARTED

Backend:
http://127.0.0.1:8000

API Testing:
http://127.0.0.1:8000/docs

Python Team completed:
✓ File upload
✓ File processing
✓ SQLite database
✓ Database management
✓ Document API
✓ Simple search
✓ Download
✓ Delete
✓ Web Team connection
✓ AI Team connection

AI Team API:
http://127.0.0.1:8001/ask

IMPORTANT:
Only TXT files are used in this easy version.
